# 01 — Dataset Audit and Split Diagnostics

This notebook consumes a versioned dataset manifest, validates schema conformance (Draft 2020-12), strictly enforces `participant_id` and `session_id` split isolation (zero leakage between `development` and `heldout`), audits exercise coverage across the canonical capability set (`bodyweight_squat`, `push_up`, `plank`, `glute_bridge`) requiring every exercise in every required split, resolves and validates each clip's authoritative `annotation_ref`, inspects the condition taxonomy distribution, and validates ground truth annotation timing integrity.

It imports reusable functions from `kinetiq_v_vision.evaluation.audit` and emits diagnostics. No private media files or bulky outputs are committed to version control.

In [ ]:
import os
from pathlib import Path

from kinetiq_v_vision.evaluation.audit import generate_dataset_audit_report

# Resolve manifest path from environment or fall back to verified synthetic CI fixture
DEFAULT_FIXTURE = Path("../fixtures/data/synthetic_manifest.json").resolve()
manifest_path = os.getenv("DATASET_MANIFEST_PATH", str(DEFAULT_FIXTURE))
print(f"Auditing manifest at: {manifest_path}")

In [ ]:
report = generate_dataset_audit_report(manifest_path)
print(report.summary_markdown())

In [ ]:
# Enforce dataset invariants programmatically
assert len(report.manifest_errors) == 0, (
    f"Manifest schema validation errors: {report.manifest_errors}"
)
assert report.split_audit["is_isolated"], (
    "Split leakage detected — "
    f"participants: {report.split_audit['leakage_participants']}, "
    f"sessions: {report.split_audit['leakage_sessions']}"
)
assert report.exercise_audit["has_full_coverage"], (
    "Every supported exercise must have coverage in every required split "
    f"(development and heldout); missing: {report.exercise_audit['missing_split_coverage']}"
)
assert report.reference_audit["is_valid"], (
    f"Annotation reference integrity failures: {report.reference_audit['reference_errors']}"
)
assert report.annotation_audit["is_valid"], (
    f"Annotation validation failures: {report.annotation_audit}"
)

print("Dataset governance audit: all invariants verified.")